# Cloud-Adapter HRC_WHU benchmark

Run in **Google Colab with a GPU runtime**. This evaluates the official binary Cloud-Adapter checkpoint trained on HRC_WHU on **all reviewed patches**: 30 dev and 15 test, the same evaluation set used for HOT, the dark-channel prior, HSV and SegFormer. Dev and test are scored separately. The checkpoint's output is a binary class map, so there is no threshold to tune on dev.

Each 1024 x 1024 RGB patch is split into sixteen non-overlapping native-resolution 256 x 256 crops to match the HRC_WHU training crop size. The runner passes uint8 BGR values through MMSegmentation's configured preprocessor, which converts to RGB and applies the configured mean/std. This differs from the Cloud-Adapter demo's per-image min-max scaling. Results are preliminary: the labels come from a single reviewer, and HRC_WHU's 0.5-15 m Google Earth imagery is closer in resolution to Maxar but not an exact sensor/domain match. The source dataset is for academic use; review the dataset terms before other uses.

To reproduce the original three-patch dev pilot instead, add `--pilot` to the run command.

In [ ]:
import subprocess
from pathlib import Path

if not Path('/content').exists():
    raise RuntimeError('This pilot is designed for Google Colab.')
print(subprocess.run(['nvidia-smi'], check=True, capture_output=True, text=True).stdout[:2000])




RuntimeError: This pilot is designed for Google Colab.

## Install inference dependencies

The Cloud-Adapter implementation depends on compiled MMCV operators, and prebuilt MMCV wheels only exist for specific PyTorch/CUDA pairs (here torch 2.1.0 + CUDA 11.8). Colab's default Python and torch are newer, so the notebook builds an isolated Python 3.10 virtual environment (`/content/mmenv`) with `uv` and runs the pilot inside it. The notebook kernel itself is not modified. Its checkpoint is about 1.32 GB; allow time for installation and download.

In [ ]:
%pip -q install uv huggingface_hub
!uv venv /content/mmenv --python 3.10
!uv pip install -q --python /content/mmenv/bin/python torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118
!uv pip install -q --python /content/mmenv/bin/python "numpy<2" pillow matplotlib openmim mmengine==0.10.7
!uv pip install -q --python /content/mmenv/bin/python mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/index.html
!uv pip install -q --python /content/mmenv/bin/python mmsegmentation==1.2.2 mmdet==3.3.0
!uv pip install -q --python /content/mmenv/bin/python ftfy regex scipy prettytable pycocotools shapely terminaltables six einops timm
!/content/mmenv/bin/python -c "import torch, mmcv, mmcv.ops; print(torch.__version__, mmcv.__version__, torch.cuda.is_available())"

In [ ]:
import subprocess
from pathlib import Path

PROJECT_REPO = 'https://github.com/Break-Through-Tech/Ursa-Space-Systems-1D-cloud-detection-in-very-high-resolution-optical-satellite-imagery.git'
PROJECT_BRANCH = 'main'
CLOUD_ADAPTER_REPO = 'https://github.com/XavierJiezou/Cloud-Adapter.git'
CLOUD_ADAPTER_COMMIT = '78c7ea50611b2381f45206a4269a2346116f4525'
PROJECT_DIR = Path('/content/ursa-cloud-benchmark')
CLOUD_ADAPTER_DIR = Path('/content/Cloud-Adapter')

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', PROJECT_BRANCH, PROJECT_REPO, str(PROJECT_DIR)], check=True)
if not CLOUD_ADAPTER_DIR.exists():
    subprocess.run(['git', 'clone', CLOUD_ADAPTER_REPO, str(CLOUD_ADAPTER_DIR)], check=True)
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CLOUD_ADAPTER_COMMIT], cwd=CLOUD_ADAPTER_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', CLOUD_ADAPTER_COMMIT], cwd=CLOUD_ADAPTER_DIR, check=True)

EVAL_ROOT = PROJECT_DIR / 'outputs' / 'reviewed_eval_set'
if not (EVAL_ROOT / 'split_manifest.csv').is_file():
    subprocess.run(['python', 'notebooks/prepare_reviewed_eval.py'], cwd=PROJECT_DIR, check=True)
print('Evaluation patches:', EVAL_ROOT)

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = 'XavierJiezou/cloud-adapter-models'
MODEL_REVISION = 'cf76783e0696f014e251f71d1b42f74b9bf0f0f6'
CHECKPOINT_NAME = 'checkpoints/cloud-adapter/hrc_whu_full_weight.pth'
CHECKPOINT = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=CHECKPOINT_NAME,
    revision=MODEL_REVISION,
)
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'cloud_adapter_benchmark'
VENV_PYTHON = '/content/mmenv/bin/python'
print('Checkpoint size (GB):', round(Path(CHECKPOINT).stat().st_size / 1e9, 2))

## Run the benchmark

The script runs inside the `/content/mmenv` environment. For each split it prints per-patch F1/IoU as it goes, then pooled precision, recall, F1 and IoU. It saves masks, comparison panels, `metrics_per_image.csv` (including `predict_seconds` per patch), `metrics_summary.csv` and `run_metadata.json` under `outputs/cloud_adapter_benchmark/{dev,test}/cloud_adapter_hrc_whu/`. Expect a few minutes on a T4 GPU.

In [ ]:
!cd {PROJECT_DIR / 'notebooks'} && MPLBACKEND=Agg {VENV_PYTHON} -m pipeline.evaluate_cloud_adapter_pilot \
    --eval-root {EVAL_ROOT} \
    --output-root {OUTPUT_DIR} \
    --cloud-adapter-root {CLOUD_ADAPTER_DIR} \
    --checkpoint {CHECKPOINT} \
    --split both \
    --label-status human_reviewed_single_reviewer
# Saved predictions and metadata are written to OUTPUT_DIR only if the run above succeeds.
print('Output directory:', OUTPUT_DIR)

## Save the results

Colab storage is wiped when the runtime resets. This zips the outputs and downloads them to your computer. Unzip into `outputs/cloud_adapter_benchmark/` in the project so the `dev/` and `test/` folders sit beside the other benchmark outputs.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/cloud_adapter_benchmark', 'zip', OUTPUT_DIR)
files.download(archive)